# Day 1 · Section 4: Text Tokenization

Standalone student notebook. Run cells in order, change the examples, and use the checks to explain what happened. It contains no workshop slides. CPU exercises work without downloads; optional Qwen3 cells require network access and, where indicated, a suitable GPU.


## Goals · 4.1–4.6

Distinguish characters, whitespace words, subword pieces and IDs; simulate BPE training/encoding; inspect Qwen3 tokenizer files and special tokens; compare five languages and calculate a context budget. The toy BPE demonstration is intentionally smaller than a production byte-level tokenizer.


In [ ]:
import re, unicodedata
from collections import Counter
text='The battery is fully charged.'
print('Unicode code points:',len(text),'whitespace words:',len(text.split()))
print('toy word/punctuation split:',re.findall(r"\w+|[^\w\s]",text))
print('UTF-8 byte count:',len(text.encode('utf-8')))
assert len(text.encode('utf-8'))>=len(text)


### 4.2 · A small BPE-like merge exercise

Count adjacent symbol pairs, pick the most frequent, then replace non-overlapping occurrences. Real Qwen3 tokenization uses learned byte-level merge rules and additional pretokenization and special-token handling.


In [ ]:
words=['lower','lowest','low','lower']
vocab=[list(word) for word in words]
def pair_counts(sequences):
    return Counter(pair for seq in sequences for pair in zip(seq,seq[1:]))
def merge_pair(seq,pair):
    out=[];i=0
    while i<len(seq):
        if i+1<len(seq) and tuple(seq[i:i+2])==pair:
            out.append(''.join(pair));i+=2
        else:out.append(seq[i]);i+=1
    return out
rules=[]
for step in range(3):
    counts=pair_counts(vocab)
    best=max(sorted(counts),key=lambda pair:counts[pair])
    rules.append(best)
    vocab=[merge_pair(seq,best) for seq in vocab]
    print('round',step+1,'merge',best,'count',counts[best],'first word',vocab[0])
assert len(rules)==3


In [ ]:
def encode_with_rules(word,learned_pairs):
    symbols=list(word)
    for pair in learned_pairs:symbols=merge_pair(symbols,pair)
    return symbols
for word in ['lower','low','lowish']:
    print(word,'→',encode_with_rules(word,rules))
print('An unseen word remains representable with smaller symbols.')


### 4.3–4.4 · Qwen tokenizer and exact round trip

On Colab, download the tokenizer (no GPU needed). `vocab_size` counts its base vocabulary, which need not equal the model's embedding-row count. Inspect IDs, pieces, special tokens and the decode result; token strings may expose byte or whitespace markers.


In [ ]:
import sys, subprocess
IN_COLAB='google.colab' in sys.modules
RUN_DOWNLOADS=IN_COLAB  # set True locally when network/package installation is intended
print('Download optional model assets:',RUN_DOWNLOADS)


In [ ]:
tokenizer=None
if RUN_DOWNLOADS:
    subprocess.check_call([sys.executable,'-m','pip','-q','install','transformers>=4.52.4,<6'])
    from transformers import AutoTokenizer
    tokenizer=AutoTokenizer.from_pretrained('Qwen/Qwen3-4B')
    sample='Battery reset.'
    ids=tokenizer.encode(sample,add_special_tokens=False)
    print('base vocabulary:',tokenizer.vocab_size,'EOS:',tokenizer.eos_token,tokenizer.eos_token_id)
    print('IDs:',ids,'pieces:',tokenizer.convert_ids_to_tokens(ids))
    print('decoded:',tokenizer.decode(ids),'round trip:',tokenizer.decode(ids)==sample)
else:print('Qwen tokenizer download skipped. Set RUN_DOWNLOADS=True for exact IDs.')


### 4.5 · Indian-language tokenization

These short sentences have approximately the same meaning; differences in script and phrasing complicate direct language comparisons. Record the observed tokenizer counts rather than assuming one word equals one token.


In [ ]:
samples={
 'Kannada':'ಬ್ಯಾಟರಿ ಸಂಪೂರ್ಣವಾಗಿ ಚಾರ್ಜ್ ಆಗಿದೆ.',
 'English':'The battery is fully charged.',
 'Hindi':'बैटरी पूरी तरह चार्ज है।',
 'Tamil':'மின்கலம் முழுமையாக சார்ஜ் ஆனது.',
 'Telugu':'బ్యాటరీ పూర్తిగా ఛార్జ్ అయింది.'}
for language,sentence in samples.items():
    count=len(tokenizer.encode(sentence,add_special_tokens=False)) if tokenizer else 'run tokenizer cell'
    print(f'{language:8} chars={len(sentence):2} words={len(sentence.split()):2} tokens={count}')
# Exercise: inspect the IDs for two scripts and decode them back.


### 4.6 · Context budgeting

Input and output tokens share a model's context capacity. A practical prompt also has formatting overhead and retrieved context. This arithmetic is illustrative; use the exact tokenizer for real budgets.


In [ ]:
context_budget=4096;reserved_output=512;prompt_overhead=128
available_for_documents=context_budget-reserved_output-prompt_overhead
print('Available input-token budget for documents:',available_for_documents)
assert available_for_documents>0
if tokenizer:
    prompt='Summarize the battery policy.'
    print('Plain-text prompt tokens (chat markers excluded):',len(tokenizer.encode(prompt,add_special_tokens=False)))


## Checks

1. Why do Unicode code points, UTF-8 bytes, words and Qwen tokens differ?
2. Encode an unfamiliar word using the toy rules. Which symbols remain unmerged?
3. Increase reserved output tokens by 256. How much room remains for retrieved text?
4. Change one multilingual sentence and compare counts without claiming tokenizer counts measure language quality.
